# Medical Model Inversion Privacy Assessment

This completed notebook evaluates confidence leakage and model inversion risk for a brain tumor MRI screening classifier using the Ultralytics brain tumor dataset.

In [1]:
from pathlib import Path
import os, sys
import torch

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.append(str(ROOT / "src"))
os.environ.setdefault("ART_DATA_PATH", str(ROOT / "art_data"))

from medical_inversion_utils import (
    art_status, evaluate_model_outputs, inversion_metrics, output_configurations,
    plot_leakage_scores, plot_reconstruction_comparison, prepare_medical_dataset,
    representative_samples, run_inversion_attack, seed_everything,
    train_or_load_model, write_csv, write_json, write_privacy_report,
)

seed_everything(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
device, art_status()

C:\Users\joshd\.conda\envs\ultralytics-env-2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


('cuda',
 {'available': True,
  'note': 'ART is installed; this exercise uses a transparent PyTorch inversion loop.'})

In [2]:
dataset = prepare_medical_dataset(ROOT / "data" / "generated")
model = train_or_load_model(ROOT / "models" / "brain_tumor_screening_cnn.pt", dataset, device=device)
baseline_metrics, confidence_rows, probs = evaluate_model_outputs(
    model, dataset.val_images, dataset.val_labels, dataset.class_names, device=device
)
write_csv(confidence_rows, ROOT / "results" / "baseline_confidence_outputs.csv")
baseline_metrics

{'accuracy': 0.49171270718232046,
 'mean_confidence': 0.8967112898826599,
 'p95_confidence': 0.9998160004615784,
 'mean_entropy': 0.24252285063266754}

In [3]:
target_classes = list(range(len(dataset.class_names)))
representatives = representative_samples(dataset.val_images, dataset.val_labels, target_classes)
reconstruction_sets = {}
metric_rows = []

for config in output_configurations():
    reconstructions = run_inversion_attack(
        model,
        target_classes,
        output_config=config,
        device=device,
        image_size=int(dataset.train_images.shape[-1]),
    )
    reconstruction_sets[config.name] = reconstructions
    metric_rows.extend(inversion_metrics(config.name, reconstructions, representatives, dataset.class_names))

write_csv(metric_rows, ROOT / "results" / "model_inversion_privacy_metrics.csv")
metric_rows[:3]

[{'output_configuration': 'full_probability',
  'target_class': np.str_('negative'),
  'model_confidence': 0.999406,
  'exposed_confidence': 0.999406,
  'predicted_class': np.str_('negative'),
  'target_match': True,
  'queries': 320,
  'representative_mse': 0.140066,
  'representative_cosine_similarity': 0.65272,
  'privacy_leakage_score': 0.65272},
 {'output_configuration': 'full_probability',
  'target_class': np.str_('positive'),
  'model_confidence': 1.0,
  'exposed_confidence': 1.0,
  'predicted_class': np.str_('positive'),
  'target_match': True,
  'queries': 320,
  'representative_mse': 0.008062,
  'representative_cosine_similarity': 0.608639,
  'privacy_leakage_score': 0.608639},
 {'output_configuration': 'rounded_confidence',
  'target_class': np.str_('negative'),
  'model_confidence': 9.1e-05,
  'exposed_confidence': 0.0,
  'predicted_class': np.str_('positive'),
  'target_match': False,
  'queries': 320,
  'representative_mse': 0.004893,
  'representative_cosine_similarity'

In [4]:
comparison_path = plot_reconstruction_comparison(
    reconstruction_sets, representatives, dataset.class_names, ROOT / "results" / "reconstructed_medical_features.png"
)
chart_path = plot_leakage_scores(metric_rows, ROOT / "results" / "privacy_leakage_by_output_config.png")
summary_path = write_json(
    {"baseline_model": baseline_metrics, "art_status": art_status(), "attack_metrics": metric_rows},
    ROOT / "results" / "privacy_assessment_summary.json",
)
report_path = write_privacy_report(
    ROOT / "results" / "privacy_assessment_report.md",
    baseline_metrics,
    metric_rows,
    chart_path,
    comparison_path,
)
comparison_path, chart_path, summary_path, report_path

(WindowsPath('C:/Users/joshd/OneDrive/Desktop/Udacity/Course/cd15148-ai-security-c2-demos-exercises/module-15-apply-model-inversion/exercise/solution/results/reconstructed_medical_features.png'),
 WindowsPath('C:/Users/joshd/OneDrive/Desktop/Udacity/Course/cd15148-ai-security-c2-demos-exercises/module-15-apply-model-inversion/exercise/solution/results/privacy_leakage_by_output_config.png'),
 WindowsPath('C:/Users/joshd/OneDrive/Desktop/Udacity/Course/cd15148-ai-security-c2-demos-exercises/module-15-apply-model-inversion/exercise/solution/results/privacy_assessment_summary.json'),
 WindowsPath('C:/Users/joshd/OneDrive/Desktop/Udacity/Course/cd15148-ai-security-c2-demos-exercises/module-15-apply-model-inversion/exercise/solution/results/privacy_assessment_report.md'))